# 🎬 OTT High Trending Prediction
## Mini Project — Statistika Probabilitas & Machine Learning
### Kelompok 1

**Tujuan:** Memprediksi apakah konten OTT (Netflix, Prime Video, Hotstar) termasuk **High Trending** atau **Not High Trending** menggunakan metode klasifikasi Machine Learning dengan penerapan **Z-Score** sebagai komponen Statistika Probabilitas.

---
| Item | Keterangan |
|---|---|
| Dataset | OTT Movies — 2.500 konten |
| Platform | Netflix · Prime Video · Hotstar |
| Target | High Trending (Z ≥ 0.5) vs Not High Trending |
| Konsep Statistik | **Z-Score** → labeling + standardisasi + interpretasi probabilistik |
| Model ML | Logistic Regression · Random Forest |

---
## 📋 Alur Pipeline
```
Dataset → EDA → Z-Score Labeling → OneHotEncoding → StandardScaler
        → Train/Test Split → Model Training → Evaluasi → Feature Importance
```

## 📦 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Semua library berhasil diimport')
print(f'   pandas     : {pd.__version__}')
print(f'   numpy      : {np.__version__}')
print(f'   matplotlib : {plt.matplotlib.__version__}')
print(f'   seaborn    : {sns.__version__}')

## 📂 2. Load Dataset

In [ ]:
df = pd.read_csv('ott_movies_clean_unique.csv')

print(f'Shape dataset  : {df.shape[0]:,} baris × {df.shape[1]} kolom')
print(f'Ukuran memori  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print()
print('5 baris pertama:')
df.head()

### 2.1 Informasi Kolom & Tipe Data

In [ ]:
print('=== INFO DATASET ===')
print(f"{'Kolom':<22} {'Tipe':<12} {'Null':>6} {'Unik':>6}")
print('-' * 50)
for col in df.columns:
    dtype = str(df[col].dtype)
    null  = df[col].isnull().sum()
    uniq  = df[col].nunique()
    print(f'{col:<22} {dtype:<12} {null:>6} {uniq:>6}')
print()
print(f'✅ Total null values : {df.isnull().sum().sum()}')

### 2.2 Statistik Deskriptif Kolom Numerik

In [ ]:
num_cols = ['rating', 'votes', 'duration_minutes',
            'engagement_score', 'popularity_score', 'trending_score']

df[num_cols].describe().round(3)

## 📊 3. Exploratory Data Analysis (EDA)
### 3.1 Distribusi Data Kategorikal

In [ ]:
cat_cols = ['type', 'platform', 'genre', 'country', 'language']
COLORS   = ['#4C6EF5', '#F59F00', '#40C057', '#FA5252', '#845EF7']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('Distribusi Data Kategorikal', fontsize=14, fontweight='bold', y=1.01)

for ax, col, color in zip(axes, cat_cols, COLORS):
    counts = df[col].value_counts()
    bars = ax.bar(counts.index, counts.values, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(col.capitalize(), fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=35, labelsize=9)
    ax.set_ylabel('Jumlah Konten')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('eda_kategorikal.png', bbox_inches='tight', dpi=130)
plt.show()
print('💡 Insight: Dataset seimbang antar platform. Genre Drama & Sci-Fi mendominasi.')

### 3.2 Distribusi Data Numerik

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Distribusi Data Numerik', fontsize=14, fontweight='bold')
axes = axes.flatten()

palette = ['#4C6EF5','#F59F00','#40C057','#FA5252','#845EF7','#15AABF']

for ax, col, color in zip(axes, num_cols, palette):
    sns.histplot(df[col], ax=ax, color=color, kde=True,
                 bins=30, alpha=0.75, edgecolor='white', linewidth=0.5)
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean={df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='black', linestyle=':', linewidth=1.5,
               label=f'Median={df[col].median():.1f}')
    ax.set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylabel('Frekuensi')

plt.tight_layout()
plt.savefig('eda_numerik.png', bbox_inches='tight', dpi=130)
plt.show()
print('💡 Insight: trending_score berdistribusi mendekati normal — cocok untuk analisis Z-Score.')

### 3.3 Korelasi Antar Fitur Numerik

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[num_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, annot_kws={'size': 11})

ax.set_title('Correlation Matrix — Fitur Numerik', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('korelasi.png', bbox_inches='tight', dpi=130)
plt.show()
print('💡 Insight: engagement_score, popularity_score, dan trending_score berkorelasi tinggi.')
print("   Ini wajar karena ketiganya mengukur 'performa' konten dari sudut berbeda.")

## 📐 4. Statistika Probabilitas — Z-Score

### Konsep Z-Score

Z-Score (atau *standard score*) mengukur **seberapa jauh suatu nilai menyimpang dari rata-rata** dalam satuan standar deviasi.

$$Z = \frac{X - \mu}{\sigma}$$

Di mana:
- $X$ = nilai data
- $\mu$ = rata-rata (mean)
- $\sigma$ = standar deviasi

**Interpretasi & Justifikasi Threshold:**
| Nilai Z | Artinya | P(Z ≥ z) |
|---|---|---|
| Z > +2 | Sangat tinggi (outlier atas) | ~2.3% |
| **Z ≥ +0.5** | **Di atas rata-rata → HIGH TRENDING** | **~30.85%** |
| -0.5 ≤ Z ≤ 0.5 | Sekitar rata-rata | ~38.3% |
| Z < -0.5 | Di bawah rata-rata | ~30.85% |

> **Justifikasi Threshold Z = 0.5:**  
> Threshold dipilih agar ≈30% konten teratas menjadi target High Trending — sesuai kebutuhan bisnis platform yang ingin memprioritaskan top-tier content. Secara statistik, P(Z ≥ 0.5) ≈ 30.85% pada distribusi normal.

### 4.1 Menghitung Z-Score `trending_score`

In [ ]:
# Hitung Z-Score secara manual (untuk transparansi akademik)
mean_ts = df['trending_score'].mean()
std_ts  = df['trending_score'].std()

df['z_score_trending'] = (df['trending_score'] - mean_ts) / std_ts

print('=== STATISTIK Z-SCORE TRENDING SCORE ===')
print(f'Mean trending_score   : {mean_ts:.4f}')
print(f'Std trending_score    : {std_ts:.4f}')
print()
print(df['z_score_trending'].describe().round(4))
print()

# Identifikasi konten berdasarkan Z-Score
outlier_high = df[df['z_score_trending'] > 2]
outlier_low  = df[df['z_score_trending'] < -2]

print(f'Konten dengan Z > +2 (performa sangat tinggi) : {len(outlier_high):>4} ({len(outlier_high)/len(df)*100:.1f}%)')
print(f'Konten dengan Z < -2 (performa sangat rendah) : {len(outlier_low):>4} ({len(outlier_low)/len(df)*100:.1f}%)')

### 4.2 Visualisasi Distribusi Z-Score + Probabilitas Normal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Analisis Z-Score — Trending Score', fontsize=14, fontweight='bold')

# ── Plot 1: Distribusi Z-Score dari data ──────────────────────────────
ax = axes[0]
z_vals = df['z_score_trending']

sns.histplot(z_vals, bins=40, ax=ax, color='#4C6EF5', kde=True,
             alpha=0.6, edgecolor='white', linewidth=0.4)

ax.axvspan(0.5,  z_vals.max(), alpha=0.15, color='#4C6EF5', label='High Trending (Z≥0.5)')
ax.axvspan(z_vals.min(), 0.5, alpha=0.12, color='#FA5252', label='Not High Trending')
ax.axvspan(2.0,  z_vals.max(), alpha=0.3,  color='#F59F00', label='Outlier atas (Z>2)')

ax.axvline(0.5, color='#FA5252', linewidth=2.2, linestyle='--', label='Threshold Z=0.5')
ax.axvline(0,   color='black',   linewidth=1.2, linestyle=':', label='Mean Z=0')
ax.axvline(2,   color='#F59F00', linewidth=1.5, linestyle='--', alpha=0.8, label='Outlier Z=2')

ax.set_xlabel('Z-Score', fontsize=12)
ax.set_ylabel('Frekuensi', fontsize=12)
ax.set_title('Distribusi Z-Score Data OTT', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# ── Plot 2: Kurva Normal + P(Z ≥ 0.5) ────────────────────────────────
ax2 = axes[1]
x = np.linspace(-4, 4, 300)
y_pdf = stats.norm.pdf(x)

ax2.plot(x, y_pdf, color='#1B2A6B', lw=2.5, label='Distribusi Normal Standar')
ax2.fill_between(x, y_pdf, where=(x >= 0.5), alpha=0.40, color='#4C6EF5',
                 label=f'P(Z ≥ 0.5) ≈ 30.85%\n→ 810 konten High Trending')
ax2.fill_between(x, y_pdf, where=(x < 0.5),  alpha=0.15, color='#ADB5BD')
ax2.axvline(0.5, color='#FA5252', linewidth=2.2, linestyle='--', label='Threshold Z=0.5')

# Annotation probabilitas
p_value = 1 - stats.norm.cdf(0.5)
ax2.annotate(f'P(Z ≥ 0.5) = {p_value:.4f}\n≈ {p_value*100:.2f}%\n(30.85 percentile dari atas)',
             xy=(1.2, 0.15), xytext=(1.8, 0.28),
             arrowprops=dict(arrowstyle='->', color='#1B2A6B', lw=1.5),
             fontsize=10, color='#1B2A6B', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                       edgecolor='#4C6EF5', alpha=0.9))

ax2.set_xlabel('Z-Score', fontsize=12)
ax2.set_ylabel('Densitas Probabilitas', fontsize=12)
ax2.set_title('Distribusi Normal Standar\nProbabilitas P(Z ≥ Threshold)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9, loc='upper right')
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('zscore_analysis.png', bbox_inches='tight', dpi=130)
plt.show()

print(f'💡 Probabilitas teoritis P(Z ≥ 0.5) = {p_value:.6f} = {p_value*100:.2f}%')
print(f'   Jumlah konten High Trending aktual  = {(df["z_score_trending"] >= 0.5).sum()}')
print(f'   Proporsi aktual                     = {(df["z_score_trending"] >= 0.5).mean()*100:.2f}%')

### 4.3 Membuat Label Target dari Z-Score

In [ ]:
# Threshold Z >= 0.5 → High Trending
THRESHOLD_Z = 0.5
df['is_high_trending'] = (df['z_score_trending'] >= THRESHOLD_Z).astype(int)

total    = len(df)
high     = df['is_high_trending'].sum()
not_high = total - high

print('=== DISTRIBUSI LABEL TARGET ===')
print(f'Threshold Z-Score  : >= {THRESHOLD_Z}')
print(f'High Trending  (1) : {high:>5} konten ({high/total*100:.1f}%)')
print(f'Not High       (0) : {not_high:>5} konten ({not_high/total*100:.1f}%)')
print(f'Total              : {total:>5} konten')

## ⚙️ 5. Feature Engineering & Preprocessing
### 5.1 Seleksi Fitur

In [ ]:
CATEGORICAL_FEATURES = ['genre', 'platform', 'country', 'language', 'type']
NUMERICAL_FEATURES   = ['rating', 'votes', 'duration_minutes',
                         'engagement_score', 'popularity_score']
TARGET = 'is_high_trending'

# PENTING: trending_score & z_score_trending DIKELUARKAN dari fitur
# Alasan: target (is_high_trending) dibuat dari trending_score → data leakage jika dipakai!
print('=== FITUR YANG DIGUNAKAN ===')
print(f'Fitur Kategorikal : {CATEGORICAL_FEATURES}')
print(f'Fitur Numerik     : {NUMERICAL_FEATURES}')
print(f'Target            : {TARGET}')
print()
print('⚠️  trending_score & z_score_trending DIKELUARKAN dari fitur')
print('   (untuk menghindari data leakage karena target dibuat dari kolom ini)')

### 5.2 Preprocessing: OneHotEncoding + StandardScaler

> **Perbaikan dari versi sebelumnya:** `LabelEncoder` diganti dengan `OneHotEncoder` untuk fitur kategorikal.  
> **Alasan:** `LabelEncoder` mengasumsikan urutan ordinal (Drama > Comedy > Action secara numerik) — **salah secara matematis** untuk fitur non-ordinal, terutama pada Logistic Regression.  
> `OneHotEncoder` mengubah tiap kategori menjadi kolom biner independen tanpa mengimplikasikan urutan.

In [ ]:
X = df[CATEGORICAL_FEATURES + NUMERICAL_FEATURES]
y = df[TARGET]

# ── Train-Test Split 80:20 (stratified) ────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('=== PEMBAGIAN DATA ===')
print(f'Train  : {len(X_train):>5} baris ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test   : {len(X_test):>5} baris ({len(X_test)/len(X)*100:.0f}%)')
print()
print('Distribusi target (train):')
vc = y_train.value_counts()
print(f'  High Trending (1) : {vc[1]} ({vc[1]/len(y_train)*100:.1f}%)')
print(f'  Not High      (0) : {vc[0]} ({vc[0]/len(y_train)*100:.1f}%)')

# ── ColumnTransformer: OHE untuk kategorikal, Scaler untuk numerik ────
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERICAL_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES)
])

# Fit pada TRAIN saja, transform pada keduanya (NO DATA LEAKAGE)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

n_features_out = X_train_proc.shape[1]
print(f'\n✅ Preprocessing selesai')
print(f'   Input features  : {X_train.shape[1]}')
print(f'   Output features : {n_features_out} (setelah OHE expansion)')
print(f'   StandardScaler  : fit pada train, transform pada test ✓')

## 🤖 6. Training Model Machine Learning
### 6.1 Logistic Regression

In [ ]:
# class_weight='balanced' untuk menangani ketidakseimbangan kelas (32% vs 68%)
lr_model = LogisticRegression(max_iter=500, random_state=42, class_weight='balanced')
lr_model.fit(X_train_proc, y_train)

y_pred_lr = lr_model.predict(X_test_proc)
y_prob_lr = lr_model.predict_proba(X_test_proc)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

print('=== LOGISTIC REGRESSION ===')
print(f'Accuracy : {acc_lr*100:.2f}%')
print(f'ROC-AUC  : {auc_lr:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_lr,
      target_names=['Not High Trending', 'High Trending']))

### 6.2 Random Forest

In [ ]:
# Random Forest tidak membutuhkan preprocessing (X_train tanpa scaling)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42,
                                   n_jobs=-1, class_weight='balanced')
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print('=== RANDOM FOREST ===')
print(f'Accuracy : {acc_rf*100:.2f}%')
print(f'ROC-AUC  : {auc_rf:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_rf,
      target_names=['Not High Trending', 'High Trending']))

## 📈 7. Evaluasi & Perbandingan Model
### 7.1 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Confusion Matrix', fontsize=13, fontweight='bold')

models_eval = [
    ('Logistic Regression', y_pred_lr, acc_lr),
    ('Random Forest',       y_pred_rf, acc_rf)
]

for ax, (name, y_pred, acc) in zip(axes, models_eval):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not High', 'High'],
                yticklabels=['Not High', 'High'],
                linewidths=1, linecolor='white', annot_kws={'size': 13, 'weight': 'bold'})
    ax.set_title(f'{name}\nAccuracy: {acc*100:.2f}%', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)

plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight', dpi=130)
plt.show()

### 7.2 ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={auc_lr:.3f})',
        color='#4C6EF5', linewidth=2.5)

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
ax.plot(fpr_rf, tpr_rf, label=f'Random Forest       (AUC={auc_rf:.3f})',
        color='#40C057', linewidth=2.5)

ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1.5, label='Baseline (AUC=0.5)')
ax.fill_between(fpr_lr, tpr_lr, alpha=0.08, color='#4C6EF5')
ax.fill_between(fpr_rf, tpr_rf, alpha=0.08, color='#40C057')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Perbandingan Model', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('roc_curve.png', bbox_inches='tight', dpi=130)
plt.show()

### 7.3 Feature Importance (Random Forest)

In [ ]:
all_features = CATEGORICAL_FEATURES + NUMERICAL_FEATURES
importances  = rf_model.feature_importances_
sorted_idx   = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
colors_fi = ['#4C6EF5' if i < 3 else '#ADB5BD' for i in range(len(all_features))]

bars = ax.bar(range(len(all_features)),
              [importances[i] for i in sorted_idx],
              color=[colors_fi[i] for i in range(len(all_features))],
              alpha=0.85, edgecolor='white')

ax.set_xticks(range(len(all_features)))
ax.set_xticklabels([all_features[i].replace('_',' ').title() for i in sorted_idx],
                   rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Feature Importance', fontsize=11)
ax.set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')

for bar, i in zip(bars, sorted_idx):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{importances[i]:.4f}', ha='center', fontsize=9, color='#1E293B')

p1 = mpatches.Patch(color='#4C6EF5', label='Top 3 Fitur')
p2 = mpatches.Patch(color='#ADB5BD', label='Fitur Lainnya')
ax.legend(handles=[p1, p2], fontsize=10)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=130)
plt.show()

print('=== TOP 5 FITUR TERPENTING ===')
for rank, i in enumerate(sorted_idx[:5], 1):
    print(f'  {rank}. {all_features[i]:<25}: {importances[i]:.4f}')

### 7.4 Perbandingan Hasil Model

In [ ]:
def get_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy'  : round(accuracy_score(y_true, y_pred) * 100, 2),
        'Precision' : round(precision_score(y_true, y_pred) * 100, 2),
        'Recall'    : round(recall_score(y_true, y_pred) * 100, 2),
        'F1-Score'  : round(f1_score(y_true, y_pred) * 100, 2),
        'ROC-AUC'   : round(roc_auc_score(y_true, y_prob), 4),
    }

results = pd.DataFrame({
    'Logistic Regression' : get_metrics(y_test, y_pred_lr, y_prob_lr),
    'Random Forest'       : get_metrics(y_test, y_pred_rf, y_prob_rf),
}).T

print('=== PERBANDINGAN METRIK MODEL ===')
print(results.to_string())
print()
best = results['Accuracy'].idxmax()
print(f'🏆 Model terbaik: {best} (Accuracy: {results.loc[best, "Accuracy"]}%)')

## 🎯 8. Demo Prediksi

Fungsi berikut mensimulasikan prediksi apakah sebuah konten OTT akan **High Trending** atau tidak.

In [ ]:
def predict_trending(genre, platform, country, language, content_type,
                     rating, votes, duration_minutes,
                     engagement_score, popularity_score, model='rf'):
    """
    Prediksi apakah konten OTT termasuk High Trending atau Not High Trending.

    Parameters
    ----------
    genre, platform, country, language, content_type : str
    rating          : float (5.0 - 9.5)
    votes           : int
    duration_minutes: int
    engagement_score: float
    popularity_score: float
    model           : 'rf' (Random Forest) atau 'lr' (Logistic Regression)
    """
    input_df = pd.DataFrame([{
        'genre'            : genre,
        'platform'         : platform,
        'country'          : country,
        'language'         : language,
        'type'             : content_type,
        'rating'           : rating,
        'votes'            : votes,
        'duration_minutes' : duration_minutes,
        'engagement_score' : engagement_score,
        'popularity_score' : popularity_score,
    }])

    if model == 'lr':
        X_proc  = preprocessor.transform(input_df)
        pred    = lr_model.predict(X_proc)[0]
        prob    = lr_model.predict_proba(X_proc)[0]
        name    = 'Logistic Regression'
    else:
        pred    = rf_model.predict(input_df)[0]
        prob    = rf_model.predict_proba(input_df)[0]
        name    = 'Random Forest'

    label  = '🔥 HIGH TRENDING' if pred == 1 else '❌ Not High Trending'
    print('=' * 52)
    print(f' PREDIKSI OTT CONTENT — {name}')
    print('=' * 52)
    print(f' Genre/Tipe     : {genre} ({content_type})')
    print(f' Platform       : {platform}')
    print(f' Negara/Bahasa  : {country} / {language}')
    print(f' Rating         : {rating} | Votes: {votes:,}')
    print(f' Durasi         : {duration_minutes} menit')
    print(f' Engagement     : {engagement_score:.1f} | Popularity: {popularity_score:.1f}')
    print('-' * 52)
    print(f' Prediksi       : {label}')
    print(f' Probabilitas   : High={prob[1]:.4f} | Not High={prob[0]:.4f}')
    print('=' * 52)
    return pred, prob

print('✅ Fungsi prediksi siap digunakan!')

In [ ]:
# Contoh 1: Drama Korea di Netflix — diprediksi HIGH TRENDING
predict_trending(
    genre='Drama', platform='Netflix', country='South Korea',
    language='Korean', content_type='Series',
    rating=9.0, votes=195000, duration_minutes=60,
    engagement_score=2800.0, popularity_score=1750.0,
    model='rf'
)

In [ ]:
# Contoh 2: Comedy India di Hotstar — diprediksi NOT HIGH TRENDING
predict_trending(
    genre='Comedy', platform='Hotstar', country='India',
    language='Hindi', content_type='Movie',
    rating=6.2, votes=25000, duration_minutes=110,
    engagement_score=120.0, popularity_score=250.0,
    model='rf'
)